# Lab 7: Model Evaluation & Metrics

In this lab, we learn to properly evaluate models on imbalanced multiclass data.

We'll cover:
- **Why accuracy alone is misleading** on imbalanced data
- **Balanced accuracy and macro-F1** - proper metrics for multiclass problems
- **Confusion matrices and per-class metrics** - diagnosing which classes fail
- **Comparing models** to Lab 5's baseline

## Part 1: Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import pytorch_lightning as pl
from pytorch_lightning import Trainer

from sklearn.metrics import (
    confusion_matrix, classification_report, 
    balanced_accuracy_score, f1_score,
    precision_score, recall_score
)

print("✓ Imports successful")

✓ Imports successful


## Part 2: Core Evaluation Functions

These functions from Lab 5 extract predictions and compute metrics on any trained model.

In [2]:
def _collect_preds_targets(model, dataloader):
    """Extract predictions and ground truth labels from a trained model."""
    model.eval()
    device = next(model.parameters()).device
    preds_all = []
    targets_all = []
    with torch.no_grad():
        for xb, yb in dataloader:
            xb = xb.to(device)
            logits = model(xb)
            preds = torch.argmax(logits, dim=1).detach().cpu().numpy()
            targets = yb.detach().cpu().numpy()
            preds_all.append(preds)
            targets_all.append(targets)
    y_pred = np.concatenate(preds_all).astype(np.int64)
    y_true = np.concatenate(targets_all).astype(np.int64)
    return y_true, y_pred

In [3]:
def _confusion_matrix(y_true, y_pred):
    """Build confusion matrix handling arbitrary class labels."""
    labels = np.unique(np.concatenate([y_true, y_pred]))
    label_to_idx = {int(lbl): i for i, lbl in enumerate(labels)}
    cm = np.zeros((len(labels), len(labels)), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[label_to_idx[int(t)], label_to_idx[int(p)]] += 1
    return labels, cm

In [4]:
def _imbalanced_metrics_from_cm(cm):
    """Compute balanced accuracy, macro-F1, and per-class metrics from confusion matrix."""
    tp = np.diag(cm).astype(np.float64)
    support = cm.sum(axis=1).astype(np.float64)
    pred_count = cm.sum(axis=0).astype(np.float64)

    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    precision = np.divide(tp, pred_count, out=np.zeros_like(tp), where=pred_count > 0)
    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros_like(tp),
        where=(precision + recall) > 0,
    )

    balanced_acc = recall.mean()
    macro_f1 = f1.mean()
    overall_acc = tp.sum() / cm.sum() if cm.sum() > 0 else 0.0
    return overall_acc, balanced_acc, macro_f1, recall, precision, f1

## Part 3: Create Synthetic Test Data

For this notebook, we'll create synthetic test data mimicking Lab 5's imbalanced distribution and good performance.

In [5]:
np.random.seed(42)

# Synthetic test set: 800 background + 20 samples each of 9 minority classes = 980 total
n_background = 800
n_minority = 20 * 9  
n_samples = n_background + n_minority
n_classes = 10

# Ground truth
y_true = np.concatenate([
    np.zeros(n_background, dtype=int),
    np.repeat(np.arange(1, n_classes), 20, axis=0)
])

# Predictions: 94% accuracy on background, 83% on minorities (matching Lab 5 performance)
y_pred = np.zeros_like(y_true)

# Background predictions
for i in range(n_background):
    if np.random.rand() < 0.94:
        y_pred[i] = 0
    else:
        y_pred[i] = np.random.randint(1, n_classes)

# Minority predictions
for i in range(n_background, len(y_true)):
    true_class = y_true[i]
    if np.random.rand() < 0.83:
        y_pred[i] = true_class
    else:
        y_pred[i] = np.random.randint(0, n_classes)

print(f"Test set: {len(y_true)} samples, {n_classes} classes")
print(f"Ground truth distribution: {np.bincount(y_true)}")
print(f"Prediction distribution:  {np.bincount(y_pred, minlength=n_classes)}")

Test set: 980 samples, 10 classes
Ground truth distribution: [800  20  20  20  20  20  20  20  20  20]
Prediction distribution:  [761  24  23  22  31  25  30  21  22  21]


## Part 4: Understanding Metrics on Imbalanced Data

When data is imbalanced (e.g., 80% background, 20% minorities), accuracy alone is misleading.

For this test set:
- **Overall Accuracy**: % of all correct predictions
- **Balanced Accuracy**: Mean recall per class (immune to imbalance)
- **Macro-F1**: Mean F1 per class

Lab 5 achieved these results on CORINE data:
- Overall Accuracy: 92.31%
- Balanced Accuracy: 83.38%
- Macro-F1: 83.76%

The ~9% gap between accuracy and balanced accuracy shows the data is imbalanced, but the model handles minorities well.

## Part 5: Compute Overall Metrics

In [6]:
labels, cm = _confusion_matrix(y_true, y_pred)
overall_acc, balanced_acc, macro_f1, recall, precision, f1 = _imbalanced_metrics_from_cm(cm)

print("Overall Metrics:")
print(f"  Overall Accuracy:  {overall_acc:.4f}")
print(f"  Balanced Accuracy: {balanced_acc:.4f}")
print(f"  Macro-F1:          {macro_f1:.4f}")
print(f"  Gap (Acc - Bal):   {overall_acc - balanced_acc:.4f}")

# Majority baseline
vals, cnts = np.unique(y_true, return_counts=True)
majority_baseline = cnts.max() / len(y_true)
print(f"\nMajority baseline:  {majority_baseline:.4f}")
print(f"Model improvement:  {balanced_acc - majority_baseline:.4f}")

Overall Metrics:
  Overall Accuracy:  0.9316
  Balanced Accuracy: 0.8746
  Macro-F1:          0.8034
  Gap (Acc - Bal):   0.0570

Majority baseline:  0.8163
Model improvement:  0.0583


## Part 6: Per-Class Metrics

In [7]:
class_names = [f"Class {i}" for i in range(n_classes)]

report = classification_report(
    y_true, y_pred,
    target_names=class_names,
    digits=4,
    zero_division=0
)

print(report)

              precision    recall  f1-score   support

     Class 0     0.9947    0.9463    0.9699       800
     Class 1     0.7083    0.8500    0.7727        20
     Class 2     0.7826    0.9000    0.8372        20
     Class 3     0.7273    0.8000    0.7619        20
     Class 4     0.5806    0.9000    0.7059        20
     Class 5     0.6800    0.8500    0.7556        20
     Class 6     0.6333    0.9500    0.7600        20
     Class 7     0.8571    0.9000    0.8780        20
     Class 8     0.6818    0.7500    0.7143        20
     Class 9     0.8571    0.9000    0.8780        20

    accuracy                         0.9316       980
   macro avg     0.7503    0.8746    0.8034       980
weighted avg     0.9449    0.9316    0.9359       980



In [8]:
cm = confusion_matrix(y_true, y_pred, labels=range(n_classes))
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar_kws={'label': 'Count'},
            xticklabels=range(n_classes), yticklabels=range(n_classes),
            ax=axes[0])
axes[0].set_title('Confusion Matrix (Raw Counts)')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='RdYlGn', vmin=0, vmax=1,
            cbar_kws={'label': 'Recall per Class'},
            xticklabels=range(n_classes), yticklabels=range(n_classes),
            ax=axes[1])
axes[1].set_title('Confusion Matrix (Normalized by True Class)')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

print("Diagonal = correct predictions per class")
print("Dark values in confusion matrix indicate accurate predictions")

Diagonal = correct predictions per class
Dark values in confusion matrix indicate accurate predictions


## Part 8: Class Distribution Analysis

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

true_dist = pd.Series(y_true).value_counts().sort_index()
pred_dist = pd.Series(y_pred).value_counts().sort_index()

for i in range(n_classes):
    if i not in true_dist.index:
        true_dist[i] = 0
    if i not in pred_dist.index:
        pred_dist[i] = 0

true_dist = true_dist.sort_index()
pred_dist = pred_dist.sort_index()

# Count plot
x_pos = np.arange(n_classes)
width = 0.35
axes[0].bar(x_pos - width/2, true_dist.values, width, label='Ground Truth', alpha=0.8)
axes[0].bar(x_pos + width/2, pred_dist.values, width, label='Predictions', alpha=0.8)
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].set_title('Class Distribution: Count')
axes[0].set_xticks(x_pos)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Percentage plot
true_pct = 100 * true_dist / true_dist.sum()
pred_pct = 100 * pred_dist / pred_dist.sum()

axes[1].bar(x_pos - width/2, true_pct.values, width, label='Ground Truth', alpha=0.8)
axes[1].bar(x_pos + width/2, pred_pct.values, width, label='Predictions', alpha=0.8)
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Percentage (%)')
axes[1].set_title('Class Distribution: Percentage')
axes[1].set_xticks(x_pos)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"{'Class':<8} {'True %':<12} {'Pred %':<12} {'Difference':<12}")
print("-" * 50)
for i in range(n_classes):
    diff = pred_pct.iloc[i] - true_pct.iloc[i]
    print(f"{i:<8} {true_pct.iloc[i]:>10.1f}% {pred_pct.iloc[i]:>10.1f}% {diff:>+10.1f}%")

Class    True %       Pred %       Difference  
--------------------------------------------------
0              81.6%       77.7%       -4.0%
1               2.0%        2.4%       +0.4%
2               2.0%        2.3%       +0.3%
3               2.0%        2.2%       +0.2%
4               2.0%        3.2%       +1.1%
5               2.0%        2.6%       +0.5%
6               2.0%        3.1%       +1.0%
7               2.0%        2.1%       +0.1%
8               2.0%        2.2%       +0.2%
9               2.0%        2.1%       +0.1%


## Part 9: Summary

The Lab 5 CNN achieved:**
- Balanced Accuracy: 83.38% 
- Macro-F1: 83.76%

When evaluating your own trained models, compare to these metrics and use the above functions to:
1. Extract predictions from your trained model
2. Build confusion matrices to see which classes are confused
3. Compute per-class metrics to identify problem areas
4. Compare class distributions to understand model biases

## Part 10: Explore Metric Combinations with Different Data Scenarios

Different data distributions, class imbalances, and model behaviors produce different metric combinations. 
This section provides flexible functions to generate synthetic scenarios and understand metric relationships.


In [10]:
def generate_scenario(
    scenario_name,
    n_classes=10,
    imbalance_type="moderate",  # "balanced", "moderate", "severe", "extreme"
    model_quality="good",  # "poor", "fair", "good", "excellent"
    error_type="random",  # "random", "systematic", "mixed"
    random_seed=42
):
    """
    Generate synthetic predictions under different data scenarios.
    
    Parameters:
    -----------
    scenario_name : str
        Name for this scenario
    n_classes : int
        Number of classes
    imbalance_type : str
        "balanced": equal class counts
        "moderate": 70% majority, rest minority (like lab 5)
        "severe": 85% majority, rest minority
        "extreme": 95% majority, rest minority
    model_quality : str
        "poor": ~60% accuracy
        "fair": ~75% accuracy
        "good": ~85% accuracy (like Lab 5)
        "excellent": ~95% accuracy
    error_type : str
        "random": errors randomly distributed across classes
        "systematic": model confuses specific class pairs
        "mixed": combination of both
    
    Returns:
    --------
    y_true, y_pred : arrays of true and predicted labels
    """
    np.random.seed(random_seed)
    
    # Define class distribution based on imbalance_type
    if imbalance_type == "balanced":
        samples_per_class = [100] * n_classes
    elif imbalance_type == "moderate":
        samples_per_class = [700] + [30] * (n_classes - 1)
    elif imbalance_type == "severe":
        samples_per_class = [850] + [15] * (n_classes - 1)
    elif imbalance_type == "extreme":
        samples_per_class = [950] + [5] * (n_classes - 1)
    
    n_total = sum(samples_per_class)
    
    # Create ground truth
    y_true = np.concatenate([
        np.full(count, cls) for cls, count in enumerate(samples_per_class)
    ])
    np.random.shuffle(y_true)
    
    # Define model accuracy based on quality
    accuracy_map = {
        "poor": 0.60,
        "fair": 0.75,
        "good": 0.85,
        "excellent": 0.95
    }
    target_acc = accuracy_map[model_quality]
    
    # Generate predictions
    y_pred = np.zeros_like(y_true)
    
    if error_type == "random":
        # Random errors uniformly distributed
        for i in range(n_total):
            if np.random.rand() < target_acc:
                y_pred[i] = y_true[i]
            else:
                y_pred[i] = np.random.randint(0, n_classes)
    
    elif error_type == "systematic":
        # Model systematically confuses specific class pairs
        confusion_pairs = [(0, 1), (2, 3), (4, 5)] if n_classes >= 6 else [(0, 1)]
        for i in range(n_total):
            if np.random.rand() < target_acc:
                y_pred[i] = y_true[i]
            else:
                # Error: pick a confused pair
                source, target = confusion_pairs[np.random.randint(0, len(confusion_pairs))]
                if y_true[i] == source:
                    y_pred[i] = target
                elif y_true[i] == target:
                    y_pred[i] = source
                else:
                    y_pred[i] = np.random.randint(0, n_classes)
    
    elif error_type == "mixed":
        # 70% random errors, 30% systematic
        for i in range(n_total):
            if np.random.rand() < target_acc:
                y_pred[i] = y_true[i]
            else:
                if np.random.rand() < 0.7:
                    # Random error
                    y_pred[i] = np.random.randint(0, n_classes)
                else:
                    # Systematic error
                    confusion_pairs = [(0, 1), (2, 3)]
                    source, target = confusion_pairs[np.random.randint(0, len(confusion_pairs))]
                    if y_true[i] == source:
                        y_pred[i] = target
                    elif y_true[i] == target:
                        y_pred[i] = source
                    else:
                        y_pred[i] = np.random.randint(0, n_classes)
    
    print(f"✓ Scenario: {scenario_name}")
    print(f"  Imbalance: {imbalance_type}, Quality: {model_quality}, Errors: {error_type}")
    print(f"  Total samples: {n_total}, Classes: {n_classes}")
    
    return y_true, y_pred, scenario_name


In [11]:
def evaluate_scenarios(scenarios_list):
    """
    Evaluate multiple scenarios and compute metrics for each.
    
    Parameters:
    -----------
    scenarios_list : list of tuples
        Each tuple: (scenario_name, y_true, y_pred)
    
    Returns:
    --------
    results_df : pandas DataFrame with metrics for each scenario
    """
    results = []
    
    for scenario_name, y_true, y_pred in scenarios_list:
        labels, cm = _confusion_matrix(y_true, y_pred)
        overall_acc, balanced_acc, macro_f1, recall, precision, f1 = _imbalanced_metrics_from_cm(cm)
        
        # Compute imbalance ratio (max / min support)
        support = cm.sum(axis=1)
        imbalance_ratio = support.max() / support.min() if support.min() > 0 else np.inf
        
        # Compute error patterns
        correct = (y_true == y_pred).sum()
        incorrect = len(y_true) - correct
        
        results.append({
            'Scenario': scenario_name,
            'Overall Acc': overall_acc,
            'Balanced Acc': balanced_acc,
            'Macro-F1': macro_f1,
            'Acc-BalAcc Gap': overall_acc - balanced_acc,
            'Imbalance Ratio': imbalance_ratio,
            'Correct': correct,
            'Incorrect': incorrect,
            'Min Class Recall': recall.min(),
            'Max Class Recall': recall.max(),
        })
    
    results_df = pd.DataFrame(results)
    return results_df


### Scenario A: Impact of Data Imbalance (same model quality)

In [12]:
# Vary imbalance, keep model quality = "good"
scenarios_a = []
for imbalance in ["balanced", "moderate", "severe", "extreme"]:
    y_true, y_pred, name = generate_scenario(
        f"{imbalance.capitalize()} Imbalance + Good Model",
        imbalance_type=imbalance,
        model_quality="good",
        error_type="random"
    )
    scenarios_a.append((f"{imbalance.capitalize()} Imbalance", y_true, y_pred))

results_a = evaluate_scenarios(scenarios_a)
print("\n" + "="*100)
print("SCENARIO A: How does DATA IMBALANCE affect metrics?")
print("="*100)
print(results_a.to_string(index=False))
print(f"\nKey insight: As data becomes more imbalanced (higher ratio),")
print(f"  Accuracy INCREASES (more correct majority predictions)")
print(f"  But Balanced Accuracy may DECREASE (minorities are harder)")
print(f"  The gap between Accuracy and Balanced Accuracy WIDENS")


✓ Scenario: Balanced Imbalance + Good Model
  Imbalance: balanced, Quality: good, Errors: random
  Total samples: 1000, Classes: 10
✓ Scenario: Moderate Imbalance + Good Model
  Imbalance: moderate, Quality: good, Errors: random
  Total samples: 970, Classes: 10
✓ Scenario: Severe Imbalance + Good Model
  Imbalance: severe, Quality: good, Errors: random
  Total samples: 985, Classes: 10
✓ Scenario: Extreme Imbalance + Good Model
  Imbalance: extreme, Quality: good, Errors: random
  Total samples: 995, Classes: 10

SCENARIO A: How does DATA IMBALANCE affect metrics?
          Scenario  Overall Acc  Balanced Acc  Macro-F1  Acc-BalAcc Gap  Imbalance Ratio  Correct  Incorrect  Min Class Recall  Max Class Recall
Balanced Imbalance     0.881000      0.881000  0.880774        0.000000         1.000000      881        119          0.840000              0.96
Moderate Imbalance     0.883505      0.881762  0.787993        0.001743        23.333333      857        113          0.766667            

### Scenario B: Impact of Model Quality (same data imbalance)

In [13]:
# Vary model quality, keep imbalance = "moderate" (like Lab 5)
scenarios_b = []
for quality in ["poor", "fair", "good", "excellent"]:
    y_true, y_pred, name = generate_scenario(
        f"Moderate Imbalance + {quality.capitalize()} Model",
        imbalance_type="moderate",
        model_quality=quality,
        error_type="random"
    )
    scenarios_b.append((f"{quality.capitalize()} Model", y_true, y_pred))

results_b = evaluate_scenarios(scenarios_b)
print("\n" + "="*100)
print("SCENARIO B: How does MODEL QUALITY affect metrics?")
print("="*100)
print(results_b.to_string(index=False))
print(f"\nKey insight: As model improves (fewer errors),")
print(f"  Both Accuracy AND Balanced Accuracy increase")
print(f"  The gap between them shrinks (model handles all classes better)")
print(f"  Macro-F1 tracks Balanced Accuracy closely")


✓ Scenario: Moderate Imbalance + Poor Model
  Imbalance: moderate, Quality: poor, Errors: random
  Total samples: 970, Classes: 10
✓ Scenario: Moderate Imbalance + Fair Model
  Imbalance: moderate, Quality: fair, Errors: random
  Total samples: 970, Classes: 10
✓ Scenario: Moderate Imbalance + Good Model
  Imbalance: moderate, Quality: good, Errors: random
  Total samples: 970, Classes: 10
✓ Scenario: Moderate Imbalance + Excellent Model
  Imbalance: moderate, Quality: excellent, Errors: random
  Total samples: 970, Classes: 10

SCENARIO B: How does MODEL QUALITY affect metrics?
       Scenario  Overall Acc  Balanced Acc  Macro-F1  Acc-BalAcc Gap  Imbalance Ratio  Correct  Incorrect  Min Class Recall  Max Class Recall
     Poor Model     0.644330      0.666762  0.491559       -0.022432        23.333333      625        345          0.533333          0.800000
     Fair Model     0.792784      0.827714  0.676375       -0.034931        23.333333      769        201          0.766667       

### Scenario C: Impact of Error Patterns (same overall accuracy)

In [14]:
# Same accuracy, different error types (severe imbalance to make differences visible)
scenarios_c = []
for error_type in ["random", "systematic", "mixed"]:
    y_true, y_pred, name = generate_scenario(
        f"Severe Imbalance + Good Model + {error_type.capitalize()} Errors",
        imbalance_type="severe",
        model_quality="good",
        error_type=error_type
    )
    scenarios_c.append((f"{error_type.capitalize()} Errors", y_true, y_pred))

results_c = evaluate_scenarios(scenarios_c)
print("\n" + "="*100)
print("SCENARIO C: How do ERROR PATTERNS affect metrics (same target accuracy)?")
print("="*100)
print(results_c.to_string(index=False))
print(f"\nKey insight: Models with same accuracy can have VERY different Balanced Acc / Macro-F1")
print(f"  Random errors → more balanced minority performance")
print(f"  Systematic errors → specific classes fail, widening class recall gap")
print(f"  This is why confusion matrices reveal patterns accuracy misses!")


✓ Scenario: Severe Imbalance + Good Model + Random Errors
  Imbalance: severe, Quality: good, Errors: random
  Total samples: 985, Classes: 10
✓ Scenario: Severe Imbalance + Good Model + Systematic Errors
  Imbalance: severe, Quality: good, Errors: systematic
  Total samples: 985, Classes: 10
✓ Scenario: Severe Imbalance + Good Model + Mixed Errors
  Imbalance: severe, Quality: good, Errors: mixed
  Total samples: 985, Classes: 10

SCENARIO C: How do ERROR PATTERNS affect metrics (same target accuracy)?
         Scenario  Overall Acc  Balanced Acc  Macro-F1  Acc-BalAcc Gap  Imbalance Ratio  Correct  Incorrect  Min Class Recall  Max Class Recall
    Random Errors     0.881218      0.881451  0.678648       -0.000233        56.666667      868        117          0.733333          1.000000
Systematic Errors     0.873096      0.841216  0.673071        0.031881        56.666667      860        125          0.733333          0.933333
     Mixed Errors     0.881218      0.848706  0.667547     

### Scenario D: Comprehensive Metric Space Visualization

Explore the 2D metric space with multiple variations to understand what combinations are typical.


In [15]:
# Create a dense grid of scenarios: all combinations of imbalance and model quality
scenarios_d = []
seed = 42
for imbalance in ["balanced", "moderate", "severe", "extreme"]:
    for quality in ["poor", "fair", "good", "excellent"]:
        y_true, y_pred, _ = generate_scenario(
            f"{imbalance} + {quality}",
            imbalance_type=imbalance,
            model_quality=quality,
            error_type="random",
            random_seed=seed
        )
        seed += 1
        scenarios_d.append((f"{imbalance}/{quality}", y_true, y_pred))

print("\n✓ Generated 16 scenarios across imbalance × quality combinations")

results_d = evaluate_scenarios(scenarios_d)

# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Plot 1: Accuracy vs Balanced Accuracy
scatter1 = axes[0, 0].scatter(
    results_d['Overall Acc'], results_d['Balanced Acc'],
    c=results_d['Imbalance Ratio'], s=100, cmap='RdYlGn_r', alpha=0.7, edgecolors='black'
)
axes[0, 0].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Perfect alignment')
axes[0, 0].set_xlabel('Overall Accuracy')
axes[0, 0].set_ylabel('Balanced Accuracy')
axes[0, 0].set_title('Accuracy vs Balanced Accuracy\n(color = imbalance ratio)')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()
plt.colorbar(scatter1, ax=axes[0, 0], label='Imbalance Ratio')

# Plot 2: Balanced Accuracy vs Macro-F1
scatter2 = axes[0, 1].scatter(
    results_d['Balanced Acc'], results_d['Macro-F1'],
    c=results_d['Overall Acc'], s=100, cmap='viridis', alpha=0.7, edgecolors='black'
)
axes[0, 1].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Perfect alignment')
axes[0, 1].set_xlabel('Balanced Accuracy')
axes[0, 1].set_ylabel('Macro-F1')
axes[0, 1].set_title('Balanced Accuracy vs Macro-F1\n(color = overall accuracy)')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()
plt.colorbar(scatter2, ax=axes[0, 1], label='Overall Accuracy')

# Plot 3: Imbalance Ratio vs Acc-BalAcc Gap
scatter3 = axes[1, 0].scatter(
    results_d['Imbalance Ratio'], results_d['Acc-BalAcc Gap'],
    c=results_d['Overall Acc'], s=100, cmap='coolwarm', alpha=0.7, edgecolors='black'
)
axes[1, 0].set_xlabel('Imbalance Ratio (log scale)')
axes[1, 0].set_ylabel('Accuracy - Balanced Accuracy Gap')
axes[1, 0].set_title('How imbalance widens the accuracy gap\n(color = overall accuracy)')
axes[1, 0].set_xscale('log')
axes[1, 0].grid(True, alpha=0.3, which='both')
plt.colorbar(scatter3, ax=axes[1, 0], label='Overall Accuracy')

# Plot 4: Min vs Max recall (class balance issue)
scatter4 = axes[1, 1].scatter(
    results_d['Min Class Recall'], results_d['Max Class Recall'],
    c=results_d['Acc-BalAcc Gap'], s=100, cmap='RdYlGn_r', alpha=0.7, edgecolors='black'
)
axes[1, 1].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Uniform recall')
axes[1, 1].set_xlabel('Min Class Recall')
axes[1, 1].set_ylabel('Max Class Recall')
axes[1, 1].set_title('Recall disparity across classes\n(color = Acc-BalAcc gap)')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()
plt.colorbar(scatter4, ax=axes[1, 1], label='Acc-BalAcc Gap')

plt.tight_layout()
plt.show()

print("\nScenario Results Summary:")
print(results_d.to_string(index=False))


✓ Scenario: balanced + poor
  Imbalance: balanced, Quality: poor, Errors: random
  Total samples: 1000, Classes: 10
✓ Scenario: balanced + fair
  Imbalance: balanced, Quality: fair, Errors: random
  Total samples: 1000, Classes: 10
✓ Scenario: balanced + good
  Imbalance: balanced, Quality: good, Errors: random
  Total samples: 1000, Classes: 10
✓ Scenario: balanced + excellent
  Imbalance: balanced, Quality: excellent, Errors: random
  Total samples: 1000, Classes: 10
✓ Scenario: moderate + poor
  Imbalance: moderate, Quality: poor, Errors: random
  Total samples: 970, Classes: 10
✓ Scenario: moderate + fair
  Imbalance: moderate, Quality: fair, Errors: random
  Total samples: 970, Classes: 10
✓ Scenario: moderate + good
  Imbalance: moderate, Quality: good, Errors: random
  Total samples: 970, Classes: 10
✓ Scenario: moderate + excellent
  Imbalance: moderate, Quality: excellent, Errors: random
  Total samples: 970, Classes: 10
✓ Scenario: severe + poor
  Imbalance: severe, Quality: 

### Scenario E: Custom Scenarios (Use Your Own Data or Parameters)

In [16]:
# OPTION 1: Generate synthetic scenarios with custom parameters
print("="*100)
print("CUSTOM SCENARIO EXAMPLES")
print("="*100)

# Example 1: Extreme imbalance with fair model
print("\n--- Example 1: Extreme data imbalance, fair model quality ---")
custom_scenario_1_true, custom_scenario_1_pred, _ = generate_scenario(
    "Extreme Imbalance + Fair Model",
    n_classes=10,
    imbalance_type="extreme",
    model_quality="fair",
    error_type="random",
    random_seed=123
)

labels, cm = _confusion_matrix(custom_scenario_1_true, custom_scenario_1_pred)
acc1, bal_acc1, f1_1, rec1, prec1, f1_arr1 = _imbalanced_metrics_from_cm(cm)

print(f"  Overall Accuracy:  {acc1:.4f}")
print(f"  Balanced Accuracy: {bal_acc1:.4f}")
print(f"  Macro-F1:          {f1_1:.4f}")
print(f"  Interpretation: A fair model on extremely imbalanced data shows")
print(f"    - High accuracy due to easy majority class")
print(f"    - Low balanced accuracy due to struggling minorities")

# Example 2: Balanced data with poor model
print("\n--- Example 2: Balanced classes, poor model quality ---")
custom_scenario_2_true, custom_scenario_2_pred, _ = generate_scenario(
    "Balanced + Poor Model",
    n_classes=10,
    imbalance_type="balanced",
    model_quality="poor",
    error_type="random",
    random_seed=124
)

labels, cm = _confusion_matrix(custom_scenario_2_true, custom_scenario_2_pred)
acc2, bal_acc2, f1_2, rec2, prec2, f1_arr2 = _imbalanced_metrics_from_cm(cm)

print(f"  Overall Accuracy:  {acc2:.4f}")
print(f"  Balanced Accuracy: {bal_acc2:.4f}")
print(f"  Macro-F1:          {f1_2:.4f}")
print(f"  Interpretation: A poor model on balanced data shows")
print(f"    - Low but consistent accuracy and balanced accuracy (small gap)")
print(f"    - All classes suffer similarly")

# Example 3: Using your own predictions
print("\n--- Example 3: Using YOUR OWN predictions (when ready) ---")
print("""
When you have your trained model and test data, replace the synthetic predictions:

# Load your model and test data
model = torch.load('path/to/model.pt')
test_loader = DataLoader(test_dataset, batch_size=32)

# Get predictions
y_true_yours, y_pred_yours = _collect_preds_targets(model, test_loader)

# Evaluate with the same functions
labels, cm = _confusion_matrix(y_true_yours, y_pred_yours)
overall_acc, balanced_acc, macro_f1, recall, precision, f1 = _imbalanced_metrics_from_cm(cm)

print(f"Your Model Results:")
print(f"  Overall Accuracy:  {overall_acc:.4f}")
print(f"  Balanced Accuracy: {balanced_acc:.4f}")
print(f"  Macro-F1:          {macro_f1:.4f}")

# Then compare with the synthetic scenarios above to understand your performance
""")

print("\n" + "="*100)
print("SUMMARY TABLE: What metric combinations tell you")
print("="*100)
summary_table = pd.DataFrame({
    'Scenario': [
        'High Acc + High BalAcc (small gap)',
        'High Acc + Lower BalAcc (large gap)',
        'Lower Acc + Lower BalAcc (small gap)',
        'High Acc-BalAcc gap + Low recall spread',
        'High Acc-BalAcc gap + High recall spread'
    ],
    'Meaning': [
        'Balanced data OR good model on imbalanced data',
        'Imbalanced data where model focuses on majority',
        'Difficult problem for model (poor generalization)',
        'Model learned majority well, minorities differently',
        'Model has systematic errors on specific minority classes'
    ],
    'Action': [
        'Model is doing well - check confusion matrix for edge cases',
        'Expected for imbalanced data - compare balanced acc to baseline',
        'Try better model or more data - check which classes are hardest',
        'Use balanced_accuracy or macro-F1 for evaluation',
        'Investigate confusion matrix - some pairs are easily confused'
    ]
})
print(summary_table.to_string(index=False))


CUSTOM SCENARIO EXAMPLES

--- Example 1: Extreme data imbalance, fair model quality ---
✓ Scenario: Extreme Imbalance + Fair Model
  Imbalance: extreme, Quality: fair, Errors: random
  Total samples: 995, Classes: 10
  Overall Accuracy:  0.7578
  Balanced Accuracy: 0.8553
  Macro-F1:          0.3053
  Interpretation: A fair model on extremely imbalanced data shows
    - High accuracy due to easy majority class
    - Low balanced accuracy due to struggling minorities

--- Example 2: Balanced classes, poor model quality ---
✓ Scenario: Balanced + Poor Model
  Imbalance: balanced, Quality: poor, Errors: random
  Total samples: 1000, Classes: 10
  Overall Accuracy:  0.6550
  Balanced Accuracy: 0.6550
  Macro-F1:          0.6551
  Interpretation: A poor model on balanced data shows
    - Low but consistent accuracy and balanced accuracy (small gap)
    - All classes suffer similarly

--- Example 3: Using YOUR OWN predictions (when ready) ---

When you have your trained model and test data, 

In [2]:
class VGG16(pl.LightningModule):
    """
    CNN classifier for square Sentinel-2 patches.

    Assumes flat input (B, h*w*in_channels) where spatial layout is
    channels-last: the flat was produced by X.reshape(N, -1) on an
    array of shape (N, h, w, in_channels).

    patch_size: spatial size (h = w), default 3 for 3x3 patches.
    in_channels: number of spectral bands, auto-detected in Cell 15.
    """

    def __init__(
        self,
        num_classes=10,
        in_channels=10,
        patch_size=3,
        lr=3e-4,
        max_epochs=60,
        class_weights=None,
    ):
        super().__init__()
        self.lr = lr
        self.in_channels = in_channels
        self.patch_size = patch_size
        self.num_classes = num_classes
        self.max_epochs = max_epochs

        self.features = nn.Sequential(
            # conv1: keep spatial size (3x3 → 3x3)
            nn.Conv2d(in_channels, 64, kernel_size=3, padding="same"),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding="same"),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 128, kernel_size=3, padding="same"),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding="same"),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(128, 256, kernel_size=3, padding="same"),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding="same"),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding="same"),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(256, 512, kernel_size=3, padding="same"),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding="same"),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding="same"),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(512, 512, kernel_size=3, padding="same"),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding="same"),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding="same"),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        # --- INSERT THIS LINE HERE ---
        self.avgpool = nn.AdaptiveAvgPool2d((7, 7))
        # -----------------------------
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(25088, 4096),
            nn.ReLU(),
            nn.Linear(4096, 1000),
            nn.ReLU(),
            nn.Linear(1000, 1000),
            nn.ReLU(),
            nn.Linear(1000, num_classes),
        )

        if class_weights is not None:
            self.register_buffer("class_weights", class_weights.float())
        else:
            self.class_weights = None

    def forward(self, x):
        x = self.features(x)

        # --- INSERT THIS LINE HERE ---
        x = self.avgpool(x)
        # -----------------------------
        
        return self.classifier(x)

    def _loss(self, logits, y):
        return F.cross_entropy(logits, y, weight=self.class_weights)

    def training_step(self, batch, _):
        x, y = batch
        y = y.long()
        logits = self(x)
        loss = self._loss(logits, y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, _):
        x, y = batch
        y = y.long()
        logits = self(x)
        loss = self._loss(logits, y)
        acc = (logits.argmax(1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)
        return loss

    def test_step(self, batch, _):
        x, y = batch
        y = y.long()
        logits = self(x)
        loss = self._loss(logits, y)
        acc = (logits.argmax(1) == y).float().mean()
        self.log("test_loss", loss, prog_bar=True)
        self.log("test_acc", acc, prog_bar=True)
        return loss

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.max_epochs)
        return {"optimizer": opt, "lr_scheduler": sched}

NameError: name 'pl' is not defined

In [1]:
from pathlib import Path
import torch

CKPT_PATH = Path("/p/scratch/training2600/<USER>/data/training_logs/Model85.ckpt")

# Must match the constructor args used during training
model = VGG16(
    lr=1e-5,
    num_classes=len(filtered_dataset.unique_labels),
    in_channels=16,
    patch_size=224,
    max_epochs=20,
)

ckpt = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(ckpt["state_dict"])
model.eval()
print(f"Loaded checkpoint: {CKPT_PATH}")
print(f"Keys in ckpt: {list(ckpt.keys())}")

NameError: name 'VGG16' is not defined